## <div style="background-color:blue; color:white; padding:10px;"> Import libraries </div>

In [53]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import scipy.sparse as sp
from sklearn.metrics import average_precision_score
from sklearn.metrics import f1_score, precision_score, recall_score

### <div style="background-color:blue; color:white; padding:10px;"> Load data </div>

In [130]:
FEATURES_CSV = "../Features/ADL/P_17_rgb_features.csv"
LABELS_CSV = "../Labels/ADL/P_17_labeled.csv"
GRAPH_PATH = "../Graph/ADL/P_17_transition_matrix.npz"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

features = pd.read_csv(FEATURES_CSV).drop(columns=["frame_id"]).values.astype(np.float32)
labels = pd.read_csv(LABELS_CSV).drop(columns=["frame_id"]).values.astype(np.float32)

P_sparse = sp.load_npz(GRAPH_PATH)
P = P_sparse.toarray().astype(np.float32)

X = torch.from_numpy(features).to(DEVICE)
X = torch.nn.functional.normalize(X, dim=1)
Y = torch.from_numpy(labels).to(DEVICE)
P_tensor = torch.from_numpy(P).to(DEVICE)
# P_tensor = torch.nn.functional.normalize(P_tensor, p=1, dim=1)

T, FEATURE_DIM = features.shape
_, NUM_CLASSES = labels.shape

print("Features:", features.shape)
print("Labels:", labels.shape)

Features: (26547, 512)
Labels: (26547, 32)


### <div style="background-color:blue; color:white; padding:10px;"> Define classifiers </div>

In [131]:
class Classifier(nn.Module):
    def __init__(self, input_dim, num_classes):  
        super().__init__()    
        self.net = nn.Sequential(     
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),    
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),       
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.net(x)

### Load Trained Model

In [132]:
model = Classifier(FEATURE_DIM, NUM_CLASSES).to(DEVICE)
model.load_state_dict(torch.load("../Model/ADL/P_17_model.pt", map_location=DEVICE))
model.eval()

C:\Users\PAWANESH\AppData\Local\Temp\ipykernel_22388\808270517.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("../Model/ADL/P_17_model.

Classifier(
  (net): Sequential(
    (0): Linear(in_features=512, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=256, out_features=32, bias=True)
  )
)

### <div style="background-color:blue; color:white; padding:10px;"> Define propagation function </div>

In [133]:
def residual_random_walk(logits, P, alpha=0.5, steps=5):
    Y = torch.sigmoid(logits)
    confidence = torch.abs(Y - 0.5) * 2
    uncertainty = 1 - confidence
    for _ in range(steps):
        Y = Y + alpha * uncertainty * (torch.matmul(P, Y) - Y)
    return Y

### <div style="background-color:blue; color:white; padding:10px;"> Evaluation function </div>

In [134]:
def evaluate(predictions, labels):
    y_true = labels
    y_pred = predictions
    threshold = 0.5
    y_pred_binary = (y_pred > threshold).astype(int)
    AP = []
    for c in range(y_true.shape[1]):
        if np.sum(y_true[:, c]) == 0:
            continue
        ap = average_precision_score(y_true[:, c], y_pred[:, c])
        AP.append(ap)
    mAP = np.mean(AP)
    f1 = f1_score(y_true, y_pred_binary, average="micro")
    precision = precision_score(y_true, y_pred_binary, average="micro")
    recall = recall_score(y_true, y_pred_binary, average="micro")
    return mAP, f1, precision, recall

### <div style="background-color:blue; color:white; padding:10px;"> A: Classifier only (baseline) </div>

In [135]:
with torch.no_grad(): 
    logits_A = model(X)
    preds_A = torch.sigmoid(logits_A).cpu().numpy()
mAP_A, f1_A, prec_A, rec_A = evaluate(preds_A, labels)
print("Variant A mAP:", mAP_A)

Variant A mAP: 0.8406345873135791


### <div style="background-color:blue; color:white; padding:10px;"> B: Classifier + Graph (no propagation) </div>

In [136]:
with torch.no_grad():
    logits_B = model(X)
    probs_B = torch.sigmoid(logits_B)
    graph_smoothed = torch.matmul(P_tensor, probs_B)
    preds_B = graph_smoothed.cpu().numpy()
mAP_B, f1_B, prec_B, rec_B = evaluate(preds_B, labels)

In [137]:
print("Variant B mAP:", mAP_B)

Variant B mAP: 0.8525684268340468


### <div style="background-color:blue; color:white; padding:10px;"> C: Full model (with label propagation) </div>

In [138]:
with torch.no_grad():
    logits_D = model(X)
#     propagated = label_propagation(logits_D, P_tensor)
    propagated = residual_random_walk(logits_D, P_tensor, alpha=0.5)
    preds_D = propagated.cpu().numpy()
mAP_D, f1_D, prec_D, rec_D = evaluate(preds_D, labels)
print("Variant D mAP:", mAP_D)

Variant D mAP: 0.8487830983257462


### <div style="background-color:blue; color:white; padding:10px;"> Save ablation results </div>

In [139]:
results = pd.DataFrame({ 
    "Variant": [
        "Baseline",
        "Fusion",
        "Graph",
        "Graph + Residual Random Walk Propagation"
    ],
    
    "mAP": [
        mAP_A,
        mAP_A,
        mAP_B,
        mAP_D
    ],
    
    "F1": [
        f1_A,
        f1_A,
        f1_B,
        f1_D
    ],
    
    "Precision": [
        prec_A,
        prec_A,
        prec_B,
        prec_D
    ],
    
    "Recall": [
        rec_A,
        rec_A,
        rec_B,
        rec_D
    ]
})
results.to_csv("../Ablations/ADL/P_17_ablation_results.csv", index=False)
print(results)

                                    Variant       mAP        F1  Precision  \
0                                  Baseline  0.840635  0.318924   0.190423   
1                                    Fusion  0.840635  0.318924   0.190423   
2                                     Graph  0.852568  0.328902   0.197570   
3  Graph + Residual Random Walk Propagation  0.848783  0.332062   0.199835   

     Recall  
0  0.980768  
1  0.980768  
2  0.981032  
3  0.981506  


### Class-wise AP table

In [140]:
class_AP = []
for c in range(NUM_CLASSES):
    y_true = labels[:, c]
    y_pred = preds_D[:, c]
    # Skip classes with no positive samples
    if np.sum(y_true) == 0:
        ap = np.nan
    else:
        ap = average_precision_score(y_true, y_pred)
    class_AP.append(ap)
class_df = pd.DataFrame({
    "Class": np.arange(NUM_CLASSES),
    "AP": class_AP,
    "Samples": labels.sum(axis=0)
})
class_df.to_csv("../Ablations/ADL/P_17_classwise_AP.csv", index=False)
print(class_df.head())

   Class        AP  Samples
0      0  0.842164   1411.0
1      1       NaN      0.0
2      2  0.766364    391.0
3      3  0.621275    751.0
4      4       NaN      0.0


In [141]:
print("\nIMPROVEMENT ANALYSIS")
print("--------------------")

print("Baseline mAP:", mAP_A)
print("Graph mAP:", mAP_B)
print("Residual RW mAP:", mAP_D)

print("Graph Improvement:", mAP_B - mAP_A)
print("Residual RW Improvement:", mAP_D - mAP_A)


IMPROVEMENT ANALYSIS
--------------------
Baseline mAP: 0.8406345873135791
Graph mAP: 0.8525684268340468
Residual RW mAP: 0.8487830983257462
Graph Improvement: 0.01193383952046767
Residual RW Improvement: 0.00814851101216707
